# Stage 3: DRT and Zarc Fitting

Computes the Distribution of Relaxation Times (Tikhonov regularization) for every
spectrum that passed Lin-KK, detects relaxation peaks, and fits an R0 + Zarc1 + … + ZarcN
equivalent circuit seeded from the DRT peaks.

Fit results include R, τ, α per Zarc element, C_eff = τ/R, conductivity σ, and
confidence intervals. Parameters are stored in session.json and exported to Excel.

**Reads:** `{sample_id}/Results/{condition}/stage2_kk.xlsx` · `{sample_id}/ISM validation/*.ism`
**Writes:** `{sample_id}/Results/{condition}/stage3_drt.xlsx`

## Quick links
- [Configuration](#Configuration): DRT λ, peak detection, Zarc bounds
- [Step 1: DRT](#Step-1:-DRT-computation-and-peak-detection): run once per session
- [Step 1b: DRT explorer](#Step-1b:-Interactive-DRT): inspect γ(τ) interactively
- [Step 2: Zarc fitting](#Step-2:-Zarc-circuit-fitting): live tuning panel + batch fit
- [Validation](#Fit-summary-and-effective-capacitance): Nyquist + C_eff + Arrhenius
- [Export](#Export): write output xlsx files


## Configuration

In [ ]:
import json
from pathlib import Path
from pipeline.interactive import select_sample, discover_conditions, discover_conditions_from_session
from pipeline.session import load_sample, update_sample

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

sample_dir   = NOTEBOOK_DIR / sample_id
results_base = sample_dir / "Results"

# Conditions with stage2_kk.xlsx; fallback to filesystem discovery
if results_base.exists():
    all_conditions = sorted([
        d.name for d in results_base.iterdir()
        if d.is_dir() and (d / "stage2_kk.xlsx").exists()
    ])
else:
    all_conditions = discover_conditions(sample_dir) or discover_conditions_from_session(_cfg)

def _read_mm(prompt_label: str, stored_m) -> float:
    stored_mm = f"{stored_m * 1e3:.3f} mm" if stored_m is not None else "not set"
    raw = input(f"{prompt_label} [mm]  stored={stored_mm}  (Enter to keep): ").strip().replace(",", ".")
    if raw:
        return float(raw) * 1e-3
    return stored_m

L_m = _read_mm("Thickness L", _cfg.get("L_m"))
D_m = _read_mm("Diameter  D", _cfg.get("D_m"))

def _update_session(**fields):
    update_sample(sample_id, **fields)

FOCUS_T         = None
FOCUS_CONDITION = None
SKIP_EXISTING   = False

# USE_SAVED_PARAMS: True resumes the calibration saved in session.json
# (normal use). False writes the values in THIS cell to session.json:
# edit below, set False, run once, then set back to True.
USE_SAVED_PARAMS = False

_p3 = _cfg.get("stage3_params", {}) if USE_SAVED_PARAMS else {}
print("Stage 3 parameter source:",
      "session.json (saved)" if _p3 else "notebook values (will overwrite session.json)")
DRT_CV_TYPE    = _p3.get("DRT_CV_TYPE",    "custom")
DRT_RBF_DER    = _p3.get("DRT_RBF_DER",    "2nd order")
DRT_SHAPE_S    = _p3.get("DRT_SHAPE_S",    0.5)
DRT_LAMBDA     = _p3.get("DRT_LAMBDA",     6.5e-6)

PEAK_MIN_PROM_DECADES = _p3.get("PEAK_MIN_PROM_DECADES", 0)
PEAK_HEIGHT_FRAC      = _p3.get("PEAK_HEIGHT_FRAC",      0.05)
PEAK_MIN_DIST_DECADES = _p3.get("PEAK_MIN_DIST_DECADES", 0.3)

# session.json stringifies dict keys; restore per-T int keys.
N_PEAKS_OVERRIDE = {
    cond: {(int(t) if isinstance(t, str) and t.isdigit() else t): v
           for t, v in d.items()}
    for cond, d in _p3.get("N_PEAKS_OVERRIDE", {}).items()
}
N_PEAKS_CAP      = _p3.get("N_PEAKS_CAP",      4)
N_PEAKS_FIXED    = _p3.get("N_PEAKS_FIXED",    {})

ZARC_INCLUDE_R0  = _p3.get("ZARC_INCLUDE_R0",  False)
ZARC_R0_MAX      = _p3.get("ZARC_R0_MAX",      200)
ZARC_R_DEC       = _p3.get("ZARC_R_DEC",       0.70)
ZARC_TAU_DEC     = _p3.get("ZARC_TAU_DEC",     0.70)
ZARC_ALPHA_INIT  = _p3.get("ZARC_ALPHA_INIT",  0.70)
ZARC_HF_WEIGHT   = _p3.get("ZARC_HF_WEIGHT",   0)
ZARC_FIX_PARAMS  = _p3.get("ZARC_FIX_PARAMS",  {})
ZARC_N_RESTARTS  = _p3.get("ZARC_N_RESTARTS",  4)
ZARC_RMSE_TOL    = _p3.get("ZARC_RMSE_TOL",    0.02)
ZARC_N_JOBS      = _p3.get("ZARC_N_JOBS",      0)   # 0 = auto (one process per CPU core)

CONDITION_PARAMS = _cfg.get("condition_params", {})
# Per-(condition, T) bounds written by the Step 2b live panel; JSON stores
# temperature keys as strings, convert back to int.
ZARC_PEAK_BOUNDS = {
    cond: {int(T): v for T, v in (d or {}).items()}
    for cond, d in (_cfg.get("zarc_peak_bounds") or {}).items()
}

_update_session(sample_id=sample_id, L_m=L_m, D_m=D_m, stage3_params={
    "DRT_CV_TYPE":            DRT_CV_TYPE,
    "DRT_RBF_DER":            DRT_RBF_DER,
    "DRT_SHAPE_S":            DRT_SHAPE_S,
    "DRT_LAMBDA":             DRT_LAMBDA,
    "PEAK_MIN_PROM_DECADES":  PEAK_MIN_PROM_DECADES,
    "PEAK_HEIGHT_FRAC":       PEAK_HEIGHT_FRAC,
    "PEAK_MIN_DIST_DECADES":  PEAK_MIN_DIST_DECADES,
    "N_PEAKS_OVERRIDE":       N_PEAKS_OVERRIDE,
    "N_PEAKS_CAP":            N_PEAKS_CAP,
    "N_PEAKS_FIXED":          N_PEAKS_FIXED,
    "ZARC_INCLUDE_R0":        ZARC_INCLUDE_R0,
    "ZARC_R0_MAX":            ZARC_R0_MAX,
    "ZARC_R_DEC":             ZARC_R_DEC,
    "ZARC_TAU_DEC":           ZARC_TAU_DEC,
    "ZARC_ALPHA_INIT":        ZARC_ALPHA_INIT,
    "ZARC_HF_WEIGHT":         ZARC_HF_WEIGHT,
    "ZARC_FIX_PARAMS":        ZARC_FIX_PARAMS,
    "ZARC_N_RESTARTS":        ZARC_N_RESTARTS,
    "ZARC_RMSE_TOL":          ZARC_RMSE_TOL,
    "ZARC_N_JOBS":            ZARC_N_JOBS,
})


In [ ]:
from pipeline.interactive import make_condition_selector
get_selected_conditions = make_condition_selector(all_conditions)

## Condition selector

Optional: click a condition or temperature to restrict processing to a single case.
Leave both as `None` in the config cell to process everything.


In [ ]:
# FOCUS selector: pick a condition and/or temperature by clicking (no config edit).
from pipeline.interactive import discover_conditions, make_focus_panel


def _apply_focus_nb03(cond, T):
    global FOCUS_CONDITION, FOCUS_T
    FOCUS_CONDITION = cond
    FOCUS_T = T


make_focus_panel(
    conditions = discover_conditions(NOTEBOOK_DIR / sample_id),
    temps      = [600, 575, 550, 525, 500, 475, 450, 425, 400],
    set_focus  = _apply_focus_nb03,
    init_cond  = FOCUS_CONDITION,
    init_T     = FOCUS_T,
)


## Import

Run once. Loads pipeline modules and resolves condition folders.

In [ ]:
%matplotlib inline

import sys
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as _stats

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism, load_csv_spectrum
from pipeline.drt import clip_spectrum, compute_drt, find_drt_peaks
from pipeline.fitting import fit_zarc, fit_to_rows, conductivity, build_circuit_string, resolve_condition_entry
from pipeline.plots import (
    apply_pub_style, COLOR_MAP, plot_nyquist_multipanel,
    plot_drt_stacked, plot_tau_arrhenius_consistency, plot_tau_tracks,
    plot_ceff_magnitude,
)

apply_pub_style()

ZARC_PEAK_BOUNDS: dict = {}  # populated by live panel

if L_m is None or D_m is None:
    raise ValueError(
        "L_m and D_m must be set in the Configuration cell "
        "(sample thickness and diameter are required for conductivity calculations)."
    )

A_m2 = np.pi * (D_m / 2) ** 2

conditions = get_selected_conditions()
if FOCUS_CONDITION is not None:
    conditions = [c for c in conditions if c == FOCUS_CONDITION]
    print(f"FOCUS_CONDITION active: {FOCUS_CONDITION}")

print(f"Sample    : {sample_id}")
print(f"Geometry  : L = {L_m*1e3:.3f} mm  |  D = {D_m*1e3:.3f} mm  |  A = {A_m2*1e6:.3f} mm²")
print(f"Conditions ({len(conditions)}):")
for c in conditions:
    print(f"  {c}")
if FOCUS_T is not None:
    print(f"\nFOCUS_T = {FOCUS_T} C: only this T will be processed")


def _resolve_zarc_params(condition: str, T_int: int) -> dict:
    cond_ov = resolve_condition_entry(CONDITION_PARAMS, condition)
    t_ov = cond_ov.get(T_int, {})
    return {
        "R_dec":      t_ov.get("R_dec",      cond_ov.get("R_dec",      ZARC_R_DEC)),
        "tau_dec":    t_ov.get("tau_dec",    cond_ov.get("tau_dec",    ZARC_TAU_DEC)),
        "alpha_init": t_ov.get("alpha_init", cond_ov.get("alpha_init", ZARC_ALPHA_INIT)),
        "hf_weight":  t_ov.get("hf_weight",  cond_ov.get("hf_weight",  ZARC_HF_WEIGHT)),
    }


## Step 1: DRT computation and peak detection

**Run this cell once.** For each (condition, T) it:
1. Loads the Lin-KK selected spectrum from `stage2_kk.xlsx`
2. Computes the DRT with the λ set in Configuration
3. Detects peaks and stores results in `_drt_results` (in memory)

Results are cached in memory. Re-run only if you change `DRT_LAMBDA` or peak detection parameters.

In [ ]:
# DRT results stored here; read by batch fit and live panel.
_drt_results = {}

# Guard: replica overrides in session.json must match the stage2 exports we
# are about to read, or the analysis silently uses a different replica.
from pipeline.utils import check_replica_overrides
for _m in check_replica_overrides(sample_dir,
                                  load_sample(sample_id).get("overrides", {}),
                                  conditions):
    print(f"[WARN] replica override out of sync: {_m}", flush=True)

# Substring matching of N_PEAKS_FIXED keys is intentional; warn once if ambiguous.
for _k in N_PEAKS_FIXED:
    _hits = [c for c in conditions if _k == c or _k in c]
    if len(_hits) > 1:
        print(f"[WARN] N_PEAKS_FIXED key '{_k}' matches {len(_hits)} selected "
              f"conditions: {_hits}. The fixed peak count applies to ALL of them.")

for condition in conditions:
    print(f"\n{'─'*64}\nCondition: {condition}\n{'─'*64}", flush=True)

    xlsx_path = sample_dir / "Results" / condition / "stage2_kk.xlsx"
    df_sel    = pd.read_excel(xlsx_path, sheet_name="Selected")

    # SKIP_EXISTING: load from file if already computed
    if SKIP_EXISTING:
        drt_path = sample_dir / "Results" / condition / "stage3_drt.xlsx"
        if drt_path.exists():
            print(f"  [SKIP] Loading DRT from {drt_path.name}", flush=True)
            df_pk = pd.read_excel(drt_path, sheet_name="Peaks")
            df_sm = pd.read_excel(drt_path, sheet_name="Summary")
            _drt_results[condition] = {}
            for T_nom in sorted(df_pk["T_nominal"].unique(), reverse=True):
                T_nom  = int(T_nom)
                pkrows = df_pk[df_pk["T_nominal"] == T_nom]
                peaks  = pkrows[["peak_id","tau","gamma_peak","R_approx",
                                 "tau_left","tau_right"]].to_dict("records")
                sm_row = df_sm[df_sm["T_nominal"] == T_nom]
                lv     = float(sm_row["lambda"].values[0]) if (
                    not sm_row.empty and "lambda" in df_sm.columns) else float("nan")
                fname  = sm_row["file"].values[0] if not sm_row.empty else ""
                if N_PEAKS_CAP is not None and len(peaks) > N_PEAKS_CAP:
                    by_R  = sorted(peaks, key=lambda p: p["R_approx"], reverse=True)
                    peaks = sorted(by_R[:N_PEAKS_CAP], key=lambda p: p["tau"])
                    for k, p in enumerate(peaks): p["peak_id"] = k + 1
                _drt_results[condition][T_nom] = {
                    "peaks": peaks, "entry": None,
                    "freq": None, "Z_re": None, "Z_im": None,
                    "lv": lv, "fname": fname, "pO2": None,
                }
            continue

    val_dir = sample_dir / "ISM validation" / condition
    csv_dir = sample_dir / "input_spectra" / condition   # CSV entry mode
    _drt_results[condition] = {}

    for _, row in df_sel.sort_values("T_nominal", ascending=False).iterrows():
        T_nom = int(row["T_nominal"])
        if FOCUS_T is not None and T_nom != FOCUS_T:
            continue

        fname = row["file"]
        f_min = row["f_min_cut"] if pd.notna(row.get("f_min_cut")) else None
        f_max = row["f_max_cut"] if pd.notna(row.get("f_max_cut")) else None

        print(f"\n  T = {T_nom} °C  |  {fname}", flush=True)

        ism_path = val_dir / fname
        if not ism_path.exists() and (csv_dir / fname).exists():
            ism_path = csv_dir / fname
        if not ism_path.exists():
            print(f"  [SKIP] Not found: {ism_path}")
            continue
        rec = (load_csv_spectrum(ism_path)
               if ism_path.suffix.lower() in (".csv", ".txt")
               else load_ism(ism_path))
        freq, Z_re, Z_im = clip_spectrum(rec.freq, rec.Z_re, rec.Z_im, f_min, f_max)
        print(f"  Points: {len(freq)}  range: {freq.max():.1f} – {freq.min():.4f} Hz", flush=True)

        print(f"  DRT ({DRT_CV_TYPE}) ...", end=" ", flush=True)
        try:
            entry = compute_drt(freq, Z_re, Z_im,
                                cv_type=DRT_CV_TYPE, rbf_der=DRT_RBF_DER,
                                shape_s=DRT_SHAPE_S, lambda_val=DRT_LAMBDA,
                                suppress_output=True)
            lv = float(np.squeeze(entry.lambda_value))
            print(f"λ = {lv:.2e}", flush=True)
        except Exception as e:
            print(f"ERROR: {e}")
            continue

        # Peak detection: 0 for PEAK_MIN_PROM_DECADES means "disabled" (height-based mode).
        # The widget already handles this with "if s_prom.value > 0 else None".
        peaks = find_drt_peaks(
            entry,
            min_height_frac  = PEAK_HEIGHT_FRAC,
            min_dist_decades = PEAK_MIN_DIST_DECADES,
            min_prom_decades = PEAK_MIN_PROM_DECADES or None,
        )

        # N_PEAKS_FIXED: enforce a fixed count for this condition across all T.
        _fixed_n = next(
            (v for k, v in N_PEAKS_FIXED.items() if k == condition or k in condition),
            None,
        )
        if _fixed_n is not None and len(peaks) != _fixed_n:
            by_R  = sorted(peaks, key=lambda p: p["R_approx"], reverse=True)
            peaks = sorted(by_R[:_fixed_n], key=lambda p: p["tau"])
            for k, p in enumerate(peaks): p["peak_id"] = k + 1
            if len(peaks) < _fixed_n:
                print(f"  [WARN] N_PEAKS_FIXED={_fixed_n} but only {len(peaks)} peaks found", flush=True)
            else:
                print(f"  [FIXED] N_peaks = {_fixed_n}", flush=True)

        # N_PEAKS_OVERRIDE: per-(condition, T) manual override.
        n_override = N_PEAKS_OVERRIDE.get(condition, {}).get(T_nom)
        if n_override is not None:
            by_R  = sorted(peaks, key=lambda p: p["R_approx"], reverse=True)
            peaks = sorted(by_R[:n_override], key=lambda p: p["tau"])
            for k, p in enumerate(peaks):
                p["peak_id"] = k + 1
            print(f"  [OVERRIDE] N_peaks = {n_override}", flush=True)
        elif N_PEAKS_CAP is not None and len(peaks) > N_PEAKS_CAP:
            by_R  = sorted(peaks, key=lambda p: p["R_approx"], reverse=True)
            peaks = sorted(by_R[:N_PEAKS_CAP], key=lambda p: p["tau"])
            for k, p in enumerate(peaks):
                p["peak_id"] = k + 1
            print(f"  [CAP] N_peaks capped to {N_PEAKS_CAP}", flush=True)

        n_peaks = len(peaks)
        print(f"  Peaks: {n_peaks}", flush=True)
        for p in peaks:
            print(f"    #{p['peak_id']}: τ={p['tau']:.3e} s  "
                  f"R_drt={p['R_approx']:.2f} Ω  "
                  f"γ_peak={p['gamma_peak']:.3f} Ω", flush=True)

        _drt_results[condition][T_nom] = {
            "entry": entry, "peaks": peaks,
            "freq": freq, "Z_re": Z_re, "Z_im": Z_im,
            "lv": lv, "fname": fname, "pO2": row.get("pO2_mean"),
        }
        gc.collect()

if FOCUS_T is not None:
    print(f"\nFOCUS_T={FOCUS_T} active; re-run after adjusting N_PEAKS_OVERRIDE.")
else:
    n_done = sum(len(v) for v in _drt_results.values())
    print(f"\n{'─'*64}")
    print(f"DRT complete: {n_done} spectra computed.")
    print("Inspect peaks in Step 1b (interactive); adjust N_PEAKS_FIXED or N_PEAKS_OVERRIDE for spurious peaks.")


### Step 1b: Interactive DRT

Inspect γ(τ) for any (condition, T). Adjust λ and peak detection thresholds live
to confirm that physical peaks are detected. Changes here do **not** affect the batch —
update `DRT_LAMBDA` in Configuration and re-run Step 1 to apply globally.

In [ ]:
# Interactive DRT explorer: recompute gamma(tau) + peaks for one (cond, T), live.
from matplotlib.figure import Figure
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _OK_DRTW = True
except Exception as _e:
    _OK_DRTW = False
    print(f"[INFO] interactive DRT needs ipywidgets ({_e}).")

if _OK_DRTW and _drt_results:
    _cs = [c for c, d in _drt_results.items() if d]
    if _cs:
        dwc = W.Dropdown(options=_cs, value=_cs[0], description="Cond:",
                         layout=W.Layout(width="420px"))
        dwT = W.Dropdown(options=sorted(_drt_results[_cs[0]], reverse=True),
                         description="T [°C]:", layout=W.Layout(width="160px"))
        s_reg = W.FloatLogSlider(value=DRT_LAMBDA, base=10, min=-6, max=-2, step=0.1,
                                 description="λ", readout_format=".1e",
                                 tooltip="Tikhonov regularisation λ: higher = smoother DRT, fewer/broader peaks",
                                 continuous_update=False, layout=W.Layout(width="380px"))
        s_prom = W.FloatSlider(value=(PEAK_MIN_PROM_DECADES or 0.05), min=0.0, max=0.5, step=0.01,
                               description="prom", readout_format=".2f",
                               tooltip="Peak log-prominence (decades): minimum local rise to be counted as a peak",
                               continuous_update=False, layout=W.Layout(width="320px"))
        s_h = W.FloatSlider(value=PEAK_HEIGHT_FRAC, min=0.0, max=0.2, step=0.01,
                            description="height", readout_format=".2f",
                            tooltip="Absolute height floor as fraction of gamma_max: rejects noise",
                            continuous_update=False, layout=W.Layout(width="320px"))
        out_drtw = W.Output()
        _drtw_suspend = [False]

        def _refresh_T_drtw(*_):
            _drtw_suspend[0] = True
            try:
                ts = sorted(_drt_results.get(dwc.value, {}), reverse=True)
                dwT.options = ts
                if ts and dwT.value not in ts:
                    dwT.value = ts[0]
            finally:
                _drtw_suspend[0] = False
        dwc.observe(_refresh_T_drtw, names="value")

        def _redraw_drtw(*_):
            if _drtw_suspend[0]:
                return
            with out_drtw:
                _clear(wait=True)
                drt = _drt_results.get(dwc.value, {}).get(int(dwT.value))
                if not drt or drt.get("freq") is None:
                    print("No cached freq/Z for this (cond, T); run Step 1 without SKIP_EXISTING.")
                    return
                entry = compute_drt(drt["freq"], drt["Z_re"], drt["Z_im"],
                                    cv_type="custom", rbf_der=DRT_RBF_DER,
                                    shape_s=DRT_SHAPE_S, lambda_val=float(s_reg.value))
                prom = float(s_prom.value) if s_prom.value > 0 else None
                peaks = find_drt_peaks(entry, min_height_frac=float(s_h.value),
                                       min_dist_decades=PEAK_MIN_DIST_DECADES, min_prom_decades=prom)
                # raw Figure: never registered with pyplot, so it can
                # neither leak into other cells nor be re-rendered by the
                # inline backend if this callback raises
                fig = Figure(figsize=(7, 4))
                ax = fig.subplots()
                color = COLOR_MAP.get(int(dwT.value), "#0066FF")
                y_max = entry.gamma.max() * 1.20 if entry.gamma.max() > 0 else 1.0
                ax.semilogx(entry.out_tau_vec, entry.gamma, "-", lw=1.6, color=color)
                ax.fill_between(entry.out_tau_vec, 0, entry.gamma, alpha=0.12, color=color)
                _ann = []
                for p in sorted(peaks, key=lambda q: q["gamma_peak"], reverse=True):
                    yc = p["gamma_peak"] + y_max * 0.09
                    for yp in _ann:
                        if abs(yc - yp) < y_max * 0.14:
                            yc = yp + y_max * 0.14
                    _ann.append(yc)
                    ax.axvline(p["tau"], color="grey", ls=":", lw=0.9, alpha=0.6)
                    ax.annotate(f"#{p['peak_id']}  τ={p['tau']:.1e}s",
                                xy=(p["tau"], p["gamma_peak"]),
                                xytext=(p["tau"], min(yc, y_max * 0.92)),
                                arrowprops=dict(arrowstyle="-", color="grey", lw=0.7),
                                fontsize=8, ha="center", va="bottom")
                ax.set_xlabel(r"$\tau$ / s"); ax.set_ylabel(r"$\gamma(\log\tau)$ / $\Omega$")
                ax.set_xlim(entry.out_tau_vec.min(), entry.out_tau_vec.max())
                ax.set_ylim(0, y_max)
                _display(fig)
                taus = ", ".join(f"{p['tau']:.1e}" for p in peaks) if peaks else ""
                print(f"λ={s_reg.value:.2e}  prom={prom}  height={s_h.value}  ->  "
                      f"{len(peaks)} peaks  tau=[{taus}]")
        for _w in (dwc, dwT, s_reg, s_prom, s_h):
            _w.observe(_redraw_drtw, names="value")

        w_apply_drt = W.Button(description="📥 Apply to batch", button_style="warning",
                               layout=W.Layout(width="200px"),
                               tooltip="Copy these λ / prom / height into the batch DRT config (Step 1)")
        _apply_lbl = W.HTML()
        def _apply_drt_params(_b):
            global DRT_LAMBDA, DRT_CV_TYPE, PEAK_MIN_PROM_DECADES, PEAK_HEIGHT_FRAC
            DRT_LAMBDA = float(s_reg.value)
            DRT_CV_TYPE = "custom"
            PEAK_MIN_PROM_DECADES = float(s_prom.value) if s_prom.value > 0 else None
            PEAK_HEIGHT_FRAC = float(s_h.value)
            _apply_lbl.value = (
                f"<b style='color:#b36b00'>Applied.</b> DRT_CV_TYPE=custom, DRT_LAMBDA={DRT_LAMBDA:.2e}, "
                f"PEAK_MIN_PROM_DECADES={PEAK_MIN_PROM_DECADES}, PEAK_HEIGHT_FRAC={PEAK_HEIGHT_FRAC}. "
                "Re-run Step 1 (batch DRT) to apply to all spectra.")
        w_apply_drt.on_click(_apply_drt_params)

        _display(W.VBox([W.HBox([dwc, dwT]), s_reg, W.HBox([s_prom, s_h]),
                         W.HBox([w_apply_drt, _apply_lbl]), out_drtw]))
        _redraw_drtw()
elif _OK_DRTW:
    print("[INFO] No DRT in memory; run Step 1 (DRT compute) first.")

## Step 2: Zarc circuit fitting

The batch cell fits R0 + Zarc1 + ... + ZarcN for all conditions and temperatures,
using the DRT peaks from Step 1 as initial guesses.

**Live tuning panel** (next cell): select a condition and temperature, adjust the
sliders, click **Re-fit**.

- **R_dec / tau_dec**: bound width (decades) around the DRT seed. Tighter = optimizer stays near the DRT peak.
- **alpha_init**: starting CPE exponent (0.5 = strong dispersion, 1.0 = ideal semicircle).
- **HF_weight**: extra emphasis on high-frequency points (0 = plain proportional, >0 pulls arcs toward HF).

Settings are saved to `session.json -> condition_params` and applied automatically when
the batch cell runs; do not edit `CONDITION_PARAMS` by hand.

Spurious peak at some temperature? Set `N_PEAKS_OVERRIDE` in Configuration and re-run Step 1.

*Symbols in the fit log: `R_i` = process resistance (diameter of the i-th Nyquist semicircle), `τ_i` = relaxation time, `α_i` = depression (1 = ideal semicircle), `C_eff = τ/R`. Derivations: README → Physical formulae.*


In [ ]:
# Reads from _drt_results (computed in Step 1); does NOT recompute DRT.
# To adjust the fit only, modify CONDITION_PARAMS and re-run ONLY this cell.
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pipeline.fitting import fit_condition_batch

# Per-peak bounds set in the Step 2b live panel; empty if that cell was not run.
try:
    ZARC_PEAK_BOUNDS
except NameError:
    ZARC_PEAK_BOUNDS = {}

all_results = {}
_stale = [c for c, p in list(CONDITION_PARAMS.items())
          if "DRT_LAMBDA" in p and abs(p["DRT_LAMBDA"] - DRT_LAMBDA) / max(abs(DRT_LAMBDA), 1e-12) > 0.05]
if _stale:
    for _sc in _stale:
        del CONDITION_PARAMS[_sc]
        ZARC_PEAK_BOUNDS.pop(_sc, None)
    print(f"[WARN] Invalidated stale condition_params for: {_stale}")
    print(f"       They were tuned with a different λ. Re-run Step 2b live panel with DRT_LAMBDA={DRT_LAMBDA:.2e}.")

# ---- pre-pass (cheap, sequential): DRT bookkeeping + plain-data fit tasks ----
_cond_tasks:  dict[str, list] = {}
_cond_static: dict[str, dict] = {}

for condition in conditions:
    if condition not in _drt_results or not _drt_results[condition]:
        print(f"\n{'─'*64}\nCondition: {condition}\n{'─'*64}")
        print("  [SKIP] No DRT in memory; run Step 1 first")
        all_results[condition] = {
            "drt_peaks": [], "drt_summary": [], "drt_spectra": [],
            "fit_peaks": [], "fit_summary": [],
        }
        continue

    xlsx_path = sample_dir / "Results" / condition / "stage2_kk.xlsx"
    df_sel    = pd.read_excel(xlsx_path, sheet_name="Selected")
    pO2_map   = {int(r["T_nominal"]): r.get("pO2_mean") for _, r in df_sel.iterrows()}

    cond_drt_peaks, cond_drt_summary, cond_drt_spectra = [], [], []
    nyq_records, skip_notes, tasks = {}, [], []

    for T_nom in sorted(_drt_results[condition].keys(), reverse=True):
        if FOCUS_T is not None and T_nom != FOCUS_T:
            continue

        drt   = _drt_results[condition][T_nom]
        peaks = drt["peaks"]
        entry = drt["entry"]
        freq  = drt["freq"]
        Z_re  = drt["Z_re"]
        Z_im  = drt["Z_im"]
        lv    = drt["lv"]
        fname = drt["fname"]
        pO2   = drt.get("pO2") or pO2_map.get(T_nom)
        T_K   = T_nom + 273.15

        if entry is not None:
            for tau_v, gamma_v in zip(entry.out_tau_vec, entry.gamma):
                cond_drt_spectra.append({
                    "condition": condition, "T_nominal": T_nom,
                    "tau": float(tau_v), "gamma": float(gamma_v),
                })
            cond_drt_summary.append({
                "condition": condition, "file": fname,
                "T_nominal": T_nom, "T_K": T_K, "pO2_mean": pO2,
                "N_peaks": len(peaks), "lambda": lv,
                "n_points_used": len(freq) if freq is not None else 0,
            })
        else:
            cond_drt_summary.append({
                "condition": condition, "file": fname,
                "T_nominal": T_nom, "T_K": T_K, "pO2_mean": pO2,
                "N_peaks": len(peaks), "lambda": lv,
            })

        for p in peaks:
            cond_drt_peaks.append({
                "condition": condition, "file": fname,
                "T_nominal": T_nom, "T_K": T_K, "pO2_mean": pO2, **p,
            })

        if not peaks:
            skip_notes.append(f"\n  T = {T_nom} °C  |  {fname}\n  [SKIP fit] No peaks.")
            continue
        if freq is None:
            skip_notes.append(f"\n  T = {T_nom} °C  |  {fname}\n"
                              f"  [SKIP fit] freq not available; remove SKIP_EXISTING and re-run Step 1")
            continue

        zarc_p  = _resolve_zarc_params(condition, T_nom)
        n_peaks = len(peaks)

        _ov_tag = ""
        if CONDITION_PARAMS:
            if T_nom in zarc_p or any(k in zarc_p for k in ("R_dec", "tau_dec", "alpha_init")):
                _ov_tag = f"  [R_dec={zarc_p['R_dec']} τ_dec={zarc_p['tau_dec']}]"

        _fix = (ZARC_FIX_PARAMS.get(condition, {}) or {}).get(T_nom)
        _pp = ZARC_PEAK_BOUNDS.get(condition, {}).get(T_nom)
        def _pp_or(name, scalar):
            if _pp and isinstance(_pp.get(name), (list, tuple)) and len(_pp[name]) == n_peaks:
                return list(_pp[name])
            return scalar

        nyq_records[T_nom] = (freq, Z_re, Z_im)
        tasks.append({
            "T_nominal":  T_nom,
            "fname":      fname,
            "ism_path":   str(sample_dir / "ISM validation" / condition / fname),
            "pO2":        pO2,
            "freq":       freq, "Z_re": Z_re, "Z_im": Z_im,
            "peaks":      peaks,
            "R_dec":      _pp_or("R_dec",     zarc_p["R_dec"]),
            "tau_dec":    _pp_or("tau_dec",   zarc_p["tau_dec"]),
            "alpha_init": zarc_p["alpha_init"],
            "alpha_min":  _pp_or("alpha_min", 0.5),
            "alpha_max":  _pp_or("alpha_max", 1.0),
            "hf_weight":  zarc_p["hf_weight"],
            "fix_params": _fix,
            "ov_tag":     _ov_tag,
        })

    _cond_tasks[condition] = tasks
    _cond_static[condition] = {
        "drt_peaks": cond_drt_peaks, "drt_summary": cond_drt_summary,
        "drt_spectra": cond_drt_spectra, "nyq_records": nyq_records,
        "skip_notes": skip_notes,
    }

# ---- fit: independent conditions run in parallel processes ----
# Warm-start is sequential per condition (T descending), so splitting by
# condition leaves every fit numerically identical to the serial loop.
_fitable = [c for c in conditions if c in _cond_tasks]
_n_jobs  = int(ZARC_N_JOBS) if ZARC_N_JOBS else (os.cpu_count() or 1)
_n_jobs  = max(1, min(_n_jobs, max(len(_fitable), 1)))
_common  = dict(include_r0=ZARC_INCLUDE_R0, r0_max=ZARC_R0_MAX,
                n_restarts=ZARC_N_RESTARTS, rmse_tol=ZARC_RMSE_TOL,
                L_m=L_m, D_m=D_m)

if _n_jobs > 1 and len(_fitable) > 1:
    import time
    from pipeline._worker import limit_blas_threads
    print(f"\nFitting {len(_fitable)} condition(s) on {_n_jobs} parallel processes...", flush=True)
    _t0, _fit_out = time.time(), {}
    with ProcessPoolExecutor(max_workers=_n_jobs,
                             initializer=limit_blas_threads) as _ex:
        _futs = {_ex.submit(fit_condition_batch, c, _cond_tasks[c], **_common): c
                 for c in _fitable}
        for _fut in as_completed(_futs):
            _c = _futs[_fut]
            _fit_out[_c] = _fut.result()
            print(f"  [{len(_fit_out)}/{len(_fitable)}] {_c}  finished at {time.time()-_t0:.0f}s", flush=True)
else:
    _fit_out = {c: fit_condition_batch(c, _cond_tasks[c], **_common) for c in _fitable}

for condition in _fitable:
    st  = _cond_static[condition]
    out = _fit_out[condition]
    print(f"\n{'─'*64}\nCondition: {condition}\n{'─'*64}", flush=True)
    for note in st["skip_notes"]:
        print(note)
    if out["log"]:
        print(out["log"], flush=True)
    all_results[condition] = {
        "drt_peaks":   st["drt_peaks"],
        "drt_summary": st["drt_summary"],
        "drt_spectra": st["drt_spectra"],
        "fit_peaks":   out["fit_peaks"],
        "fit_summary": out["fit_summary"],
        "nyq_records": st["nyq_records"],
        "nyq_fits":    out["nyq_fits"],
    }

gc.collect()

if CONDITION_PARAMS:
    _update_session(condition_params=CONDITION_PARAMS)

n_conv = sum(sum(1 for r in data["fit_summary"] if r.get("converged"))
             for data in all_results.values())
n_tot  = sum(len(data["fit_summary"]) for data in all_results.values())
if FOCUS_T is not None:
    print(f"\nFOCUS_T={FOCUS_T} active: only T={FOCUS_T}°C processed.")
else:
    print(f"\n{'─'*64}")
    print(f"Fitting complete: {n_conv}/{n_tot} converged.")
    print("Check Nyquist plots and the validation dashboard below.")

In [ ]:
# Bound-constraint check: is each fitted R_i / tau_i data-driven or pinned to
# the edge of its constraint window? Rebuilds the exact bounds the fit saw
# (same build_bounds call, incl. per-peak overrides). A pinned tau usually
# means the DRT seed is off (act on DRT_LAMBDA) or tau_dec is too tight
# (widen it in the Step 2b panel), then re-fit the condition.
from pipeline.fitting import build_bounds

PIN_MARGIN = 0.15   # fraction of the log half-width considered "at the edge"

def _edge_frac(v: float, lo: float, hi: float) -> float:
    """Distance of v from the nearer bound as a fraction of the log10 half-width (0 = on the bound, 1 = window center)."""
    log_lo, log_hi, log_v = np.log10(lo), np.log10(hi), np.log10(v)
    half = (log_hi - log_lo) / 2
    return float(min(log_v - log_lo, log_hi - log_v) / half) if half > 0 else 0.0

_rows = []
for _cond in _fitable:
    _fit_by_T = {}
    for _r in all_results[_cond]["fit_peaks"]:
        _fit_by_T.setdefault(int(_r["T_nominal"]), []).append(_r)
    for _task in _cond_tasks[_cond]:
        _T = int(_task["T_nominal"])
        _lo, _hi = build_bounds(0.0, _task["peaks"],
                                R_dec=_task["R_dec"], tau_dec=_task["tau_dec"],
                                alpha_min=_task["alpha_min"], alpha_max=_task["alpha_max"],
                                include_r0=False)
        for _i, _r in enumerate(sorted(_fit_by_T.get(_T, []), key=lambda r: r["peak_id"])):
            if 3 * _i + 1 >= len(_lo):
                break   # fitted peaks and DRT task peaks out of sync; do not misattribute bounds
            _fR   = _edge_frac(_r["R_i"],   _lo[3 * _i],     _hi[3 * _i])
            _ftau = _edge_frac(_r["tau_i"], _lo[3 * _i + 1], _hi[3 * _i + 1])
            _rows.append({"condition": _cond, "T": _T, "peak": int(_r["peak_id"]),
                          "R_edge_frac": round(_fR, 3), "tau_edge_frac": round(_ftau, 3),
                          "flag": "PINNED" if min(_fR, _ftau) <= PIN_MARGIN else ""})

if _rows:
    _df_bounds = pd.DataFrame(_rows)
    _n_pin = int((_df_bounds["flag"] == "PINNED").sum())
    print(f"Bound-constraint check: {_n_pin}/{len(_df_bounds)} fits pinned "
          f"(edge_frac <= {PIN_MARGIN})")
    if _n_pin:
        display(_df_bounds[_df_bounds["flag"] == "PINNED"]
                .sort_values(["condition", "T", "peak"]).reset_index(drop=True))
else:
    print("Bound-constraint check: no fits in memory (run Step 2 first)")

In [ ]:
import json
from pipeline.fitting import build_bounds

def _edge_frac(v: float, lo: float, hi: float) -> float:
    """Distance of v from the nearer bound as a fraction of the log10 half-width (0 = on the bound, 1 = window center)."""
    log_lo, log_hi, log_v = np.log10(lo), np.log10(hi), np.log10(v)
    half = (log_hi - log_lo) / 2
    return float(min(log_v - log_lo, log_hi - log_v) / half) if half > 0 else 0.0
from matplotlib.figure import Figure
# Live tuning panel.
# Changing condition/T loads current CONDITION_PARAMS values into the sliders.
# Re-fit writes slider values back to CONDITION_PARAMS (picked up by batch cell).
try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    _HAS_WIDGETS_NB03 = True
except Exception as e:
    _HAS_WIDGETS_NB03 = False
    print(f"[INFO] ipywidgets not available ({e}).")

try:
    ZARC_PEAK_BOUNDS
except NameError:
    ZARC_PEAK_BOUNDS = {}

from pipeline.fitting import resolve_condition_entry

def _load_cond_params(cond):
    """Return fitting params for a condition from CONDITION_PARAMS or global defaults."""
    return resolve_condition_entry(CONDITION_PARAMS, cond)

if _HAS_WIDGETS_NB03 and _drt_results:
    _conds = [c for c, d in _drt_results.items() if d]
    if _conds:
        _c0 = _conds[0]
        _T0 = sorted(_drt_results[_c0].keys(), reverse=True)[0]

        w_c  = W.Dropdown(options=_conds, value=_c0, description="Cond:",
                          layout=W.Layout(width="440px"))
        w_T  = W.Dropdown(options=sorted(_drt_results[_c0].keys(), reverse=True),
                          value=_T0, description="T [°C]:", layout=W.Layout(width="180px"))
        w_Rd = W.FloatSlider(value=ZARC_R_DEC,     min=0.1, max=3.0, step=0.1,
                             description="R_dec",   readout_format=".1f",
                             continuous_update=False, layout=W.Layout(width="360px"))
        w_Td = W.FloatSlider(value=ZARC_TAU_DEC,   min=0.1, max=3.0, step=0.1,
                             description="τ_dec",   readout_format=".1f",
                             continuous_update=False, layout=W.Layout(width="360px"))
        w_al = W.FloatSlider(value=ZARC_ALPHA_INIT, min=0.5, max=1.0, step=0.05,
                             description="α_init",  readout_format=".2f",
                             continuous_update=False, layout=W.Layout(width="360px"))
        w_hf = W.FloatSlider(value=ZARC_HF_WEIGHT,  min=0.0, max=2.0, step=0.1,
                             description="HF_weight", readout_format=".1f",
                             continuous_update=False, layout=W.Layout(width="360px"))
        w_go = W.Button(description="↻  Re-fit", button_style="primary",
                        layout=W.Layout(width="130px"))
        out_n = W.Output()

        _suspend = [False]

        def _load_sliders(*_):
            _suspend[0] = True
            try:
                cond = w_c.value
                p = _load_cond_params(cond)
                T  = int(w_T.value)
                tp = p.get(T, {})
                w_Rd.value = tp.get("R_dec",      p.get("R_dec",      ZARC_R_DEC))
                w_Td.value = tp.get("tau_dec",    p.get("tau_dec",    ZARC_TAU_DEC))
                w_al.value = tp.get("alpha_init", p.get("alpha_init", ZARC_ALPHA_INIT))
                w_hf.value = tp.get("hf_weight",  p.get("hf_weight",  ZARC_HF_WEIGHT))
            finally:
                _suspend[0] = False

        def _refresh_Ts(*_):
            ts = sorted(_drt_results.get(w_c.value, {}).keys(), reverse=True)
            w_T.options = ts
            if ts and w_T.value not in ts:
                w_T.value = ts[0]
            _load_sliders()

        w_c.observe(_refresh_Ts,  names="value")
        w_T.observe(_load_sliders, names="value")

        def _on_refit(_=None):
            if _suspend[0]:
                return
            cond, T = w_c.value, int(w_T.value)
            drt = _drt_results.get(cond, {}).get(T)
            if drt is None or drt.get("freq") is None:
                with out_n: print("[WARN] no DRT data — run Step 1 first.")
                return
            peaks = [dict(p) for p in drt["peaks"]]
            if not peaks:
                with out_n: print("[WARN] no peaks detected.")
                return
            n   = len(peaks)
            Rd  = float(w_Rd.value)
            Td  = float(w_Td.value)
            al  = float(w_al.value)
            hfw = float(w_hf.value)

            # Write back to CONDITION_PARAMS so batch cell picks up the change
            short = cond
            CONDITION_PARAMS.setdefault(short, {}).update(
                {"R_dec": Rd, "tau_dec": Td, "alpha_init": al, "hf_weight": hfw, "DRT_LAMBDA": DRT_LAMBDA}
            )
            ZARC_PEAK_BOUNDS.setdefault(cond, {})[T] = {
                "R_dec": [Rd]*n, "tau_dec": [Td]*n,
                "alpha_min": [0.5]*n, "alpha_max": [1.0]*n,
                "alpha_init": al,
            }

            fit = fit_zarc(drt["freq"], drt["Z_re"], drt["Z_im"], peaks,
                           R_dec=Rd, tau_dec=Td, alpha_init=al,
                           alpha_min=0.5, alpha_max=1.0,
                           include_r0=ZARC_INCLUDE_R0, r0_max=ZARC_R0_MAX,
                           hf_weight=hfw)
            with out_n:
                clear_output(wait=True)
                status = "OK" if fit["converged"] else "NOT CONVERGED"
                print(f"N={n}  {status}  rmse={fit['rmse_rel']:.4f}  R0={fit['R0']:.3g} Ω")
                for i in range(n):
                    print(f"  Zarc{i+1}: R={fit['R'][i]:.3g} Ω  "
                          f"τ={fit['tau'][i]:.2e} s  α={fit['alpha'][i]:.3f}  "
                          f"C={fit['C_eff'][i]:.2e} F")
                _lo, _hi = build_bounds(0.0, peaks, R_dec=Rd, tau_dec=Td,
                                        alpha_min=0.5, alpha_max=1.0,
                                        include_r0=False)
                print("  bound check (edge_frac: 0 = on the bound, 1 = window center):")
                for i in range(n):
                    _fR   = _edge_frac(fit["R"][i],   _lo[3 * i],     _hi[3 * i])
                    _ftau = _edge_frac(fit["tau"][i], _lo[3 * i + 1], _hi[3 * i + 1])
                    _tag  = "  <-- PINNED" if min(_fR, _ftau) <= 0.15 else ""
                    print(f"  Zarc{i + 1}: R_edge={_fR:.2f}  tau_edge={_ftau:.2f}{_tag}")

                # persist to session.json (keeps condition data out of the notebook)
                _update_session(condition_params=CONDITION_PARAMS,
                                zarc_peak_bounds=ZARC_PEAK_BOUNDS)
                print(f"\nSaved → session.json condition_params[\"{short}\"]")
                freq = drt["freq"]
                Z_exp = drt["Z_re"] - 1j * drt["Z_im"]
                Z_fit = fit["Z_fit"]
                mod   = np.abs(Z_exp)
                rr    = (Z_fit.real - Z_exp.real) / mod * 100
                ri    = (Z_fit.imag - Z_exp.imag) / mod * 100

                # raw Figure (not plt.figure): exception-safe, cannot leak
                fig = Figure(figsize=(11, 5), layout="constrained")
                gs  = fig.add_gridspec(2, 2, width_ratios=[1.25, 1],
                                       hspace=0.10, wspace=0.30)
                axN  = fig.add_subplot(gs[:, 0])
                axRr = fig.add_subplot(gs[0, 1])
                axRi = fig.add_subplot(gs[1, 1], sharex=axRr)

                sc = axN.scatter(drt["Z_re"] / 1e3, drt["Z_im"] / 1e3,
                                 c=np.log10(freq), cmap="viridis",
                                 s=26, zorder=2, linewidth=0)
                axN.plot(Z_fit.real / 1e3, -Z_fit.imag / 1e3, "-",
                         color="crimson", lw=1.1, zorder=3, label="Zarc fit")
                axN.set_xlabel(r"$Z'$ / k$\Omega$")
                axN.set_ylabel(r"$-Z''$ / k$\Omega$")
                # square window from the origin: same span on both axes, so
                # the semicircle geometry is never visually distorted
                _m = 1.05 * max(float(np.max(drt["Z_re"])), float(np.max(drt["Z_im"])),
                                float(np.max(Z_fit.real)), float(np.max(-Z_fit.imag))) / 1e3
                axN.set_xlim(0, _m)
                axN.set_ylim(0, _m)
                axN.set_aspect("equal", "box")
                axN.legend(loc="lower right", frameon=False, fontsize=10)
                axN.set_title("Nyquist with Zarc fit")
                cb = fig.colorbar(sc, ax=axN, fraction=0.046, pad=0.04)
                cb.set_label(r"$\log_{10}(f\,/\,\mathrm{Hz})$")

                for axr, resd, lab in [
                    (axRr, rr, r"$\Delta Z'$ %"),
                    (axRi, ri, r"$\Delta Z''$ %"),
                ]:
                    axr.axhline(0, color="grey", lw=0.6)
                    axr.scatter(freq, resd, c=np.log10(freq),
                                cmap="viridis", s=14, linewidth=0)
                    axr.set_xscale("log"); axr.set_ylabel(lab)
                plt.setp(axRr.get_xticklabels(), visible=False)
                axRi.set_xlabel("f / Hz")
                axRr.set_title("Relative residuals")
                display(fig)

        w_go.on_click(_on_refit)
        _load_sliders()
        display(W.VBox([W.HBox([w_c, w_T]), w_Rd, w_Td, w_al, w_hf, w_go, out_n]))
elif not _drt_results:
    print("[INFO] No DRT in memory — run Step 1 first.")

## Troubleshooting fits

| Symptom | Fix |
|---|---|
| `rmse > 0.05`, DRT looks clean | Widen bounds: increase `R_dec` or `τ_dec` in the live panel |
| Middle peaks swap between temperatures | Narrow `τ_dec` to keep each arc near its DRT seed |
| Fit shifted toward low frequency | Increase `HF_weight` (try 0.3 → 0.5) |
| Fit misses the bulk arc entirely | Check that `f_max_cut` in stage2 is not too aggressive |
| Peak count wrong at one T | Use `N_PEAKS_OVERRIDE` in Configuration and re-run Step 1 |

Per-condition overrides are stored in `session.json` (never committed to git).


## Fit summary and effective capacitance

One figure per condition:
- **Left**: 3×3 Nyquist panels (data + fit) for all 9 temperatures, labeled with rmse
- **Top right**: C_eff tracks — use to assign processes (bulk ~pF, GB ~nF, electrode ~µF)
- **Bottom right**: Arrhenius ln(τ) vs 1000/T — R² shown in legend; R² ≥ 0.97 = thermally activated process

*`σ_i = L/(R_i·A)` (conductivity of process i, transport processes only); `C_eff,i = τ_i/R_i` exact for the Zarc parametrisation. Derivations: README → Physical formulae.*


In [ ]:
# Consolidated validation: 3×3 Nyquist small-multiples + C_eff tracks + Arrhenius τ.
# One figure per condition.  Replaces the former separate Nyquist / C_eff / Arrhenius cells.
from scipy import stats as _stats
from pipeline.plots import apply_pub_style, PEAK_COLORS, PEAK_MARKERS, COLOR_MAP
apply_pub_style()

for condition, data in all_results.items():
    if not data.get("fit_peaks"):
        continue
    short  = condition.replace(sample_id + "_B_", "")
    df     = pd.DataFrame(data["fit_peaks"]).sort_values("T_nominal")
    nyq_r  = data.get("nyq_records", {})
    nyq_f  = data.get("nyq_fits",    {})

    fig = plt.figure(figsize=(14, 7), dpi=100)
    gs  = fig.add_gridspec(2, 2, width_ratios=[1.4, 1], hspace=0.40, wspace=0.34)
    gs_nyq  = gs[:, 0].subgridspec(3, 3, hspace=0.10, wspace=0.10)
    ax_ceff = fig.add_subplot(gs[0, 1])
    ax_arr  = fig.add_subplot(gs[1, 1])

    # Pre-allocate all 9 axes at once — avoids a full layout pass per subplot.
    nyq_axes = gs_nyq.subplots(sharex=False, sharey=False)

    # ── 3×3 Nyquist panels ──────────────────────────────────────────────────
    temps_desc = sorted(nyq_r.keys(), reverse=True)
    for idx in range(9):
        row, col = divmod(idx, 3)
        ax = nyq_axes[row, col]
        if idx >= len(temps_desc):
            ax.set_visible(False)
            continue
        T = temps_desc[idx]
        freq, zr, zi = nyq_r[T]
        color = COLOR_MAP.get(T, "gray")
        ax.plot(zr / 1e3, zi / 1e3, "o", color=color, ms=2.5, alpha=0.85, zorder=2)
        fit_d = nyq_f.get(T, {})
        if "Z_fit" in fit_d:
            Zf = fit_d["Z_fit"]
            ax.plot(Zf.real / 1e3, -Zf.imag / 1e3, "-", color="black", lw=0.9, zorder=3)
        _rmse = fit_d.get("rmse_rel")
        rmse_lbl = f" r={_rmse:.2f}" if _rmse is not None else ""
        _tc = ("#2ecc71" if _rmse is not None and _rmse < 0.05
               else "#e67e22" if _rmse is not None and _rmse < 0.10
               else "#e74c3c")
        ax.set_title(f"{T}°C{rmse_lbl}", fontsize=6.5, pad=2, color=_tc)
        # set_box_aspect avoids a full figure relayout per call (unlike set_aspect("equal")).
        ax.set_box_aspect(1)
        ax.tick_params(labelsize=5, direction="in", length=2, top=True, right=True)
        if row < 2: ax.set_xticklabels([])
        if col > 0: ax.set_yticklabels([])
        if row == 2: ax.set_xlabel("Z' / kΩ", fontsize=6)
        if col == 0: ax.set_ylabel("−Z'' / kΩ", fontsize=6)

    # ── C_eff tracks ─────────────────────────────────────────────────────────
    for i, (pid, g) in enumerate(df.groupby("peak_id")):
        g = g.sort_values("T_nominal")
        ax_ceff.semilogy(
            g["T_nominal"], g["C_eff_i"],
            PEAK_MARKERS[i % len(PEAK_MARKERS)] + "-",
            color=PEAK_COLORS[i % len(PEAK_COLORS)],
            ms=5, lw=1.2, label=f"P{int(pid)}",
            markeredgecolor="black", markeredgewidth=0.4,
        )
    ax_ceff.set_xlabel("T / °C", fontsize=9)
    ax_ceff.set_ylabel(r"$C_\mathrm{eff}$ / F", fontsize=9)
    ax_ceff.legend(fontsize=7, frameon=False, ncol=3, loc="best")
    ax_ceff.grid(True, which="both", ls=":", alpha=0.4)
    ax_ceff.tick_params(direction="in", top=True, right=True)

    # ── Arrhenius τ (ln τ vs 1000/T, one line+markers per peak) ─────────────────
    for i, (pid, g) in enumerate(df.groupby("peak_id")):
        g    = g.sort_values("T_nominal")
        T_K  = g["T_nominal"].values + 273.15
        invT = 1000.0 / T_K
        lnt  = np.log(g["tau_i"].values)
        col  = PEAK_COLORS[i % len(PEAK_COLORS)]
        mrk  = PEAK_MARKERS[i % len(PEAK_MARKERS)]
        r2_lbl = ""
        if len(invT) >= 3:
            sl, ic, r, *_ = _stats.linregress(invT, lnt)
            r2_lbl = f"  R²={r**2:.3f}"
            xf = np.linspace(invT.min(), invT.max(), 50)
            ax_arr.plot(xf, sl * xf + ic, "-", color=col, lw=1.0, alpha=0.6)
        ax_arr.plot(invT, lnt, mrk, color=col, ms=5,
                    markeredgecolor="black", markeredgewidth=0.4,
                    label=f"P{int(pid)}{r2_lbl}")
    ax_arr.set_xlabel(r"$1000\,T^{-1}$ / K$^{-1}$", fontsize=9)
    ax_arr.set_ylabel(r"$\ln(\tau\,/\,\mathrm{s})$", fontsize=9)
    ax_arr.legend(fontsize=7, frameon=False, ncol=2, loc="best")
    ax_arr.grid(True, which="both", ls=":", alpha=0.4)
    ax_arr.tick_params(direction="in", top=True, right=True)

    fig.suptitle(short, fontsize=10)
    plt.show()
    plt.close(fig)


In [ ]:
# Stacked DRT: visual peak shift with T (normalized curve per temperature)
# Peaks must shift rightward (larger τ) as T decreases; thermally activated behaviour
# Diagnostic preview only; save=False, so no files are written here
# (the final saved figures are produced in 04_plots.ipynb).

for condition, data in all_results.items():
    if not data["drt_spectra"]:
        continue
    df_sp = pd.DataFrame(data["drt_spectra"])
    print(f"\nDRT Stacked: condition: {condition}")
    fig = plot_drt_stacked(
        df_sp,
        condition=condition,
        save_dir=Path("."),
        tau_max=1.0,
        offset_step=1.2,
        save=False,
    )
    plt.show()
    plt.close(fig)


## Export

Writes results to the `Results/` folder for each condition:
- `stage3_drt.xlsx` — DRT peaks, summary, full γ(τ) spectra
- `stage3_fit.xlsx` — Zarc fit parameters (R, τ, α, C_eff, σ) per peak per T

Run after the batch fit. Safe to re-run (overwrites existing files).


In [ ]:
from pipeline.utils import merge_sheet_by_T, build_metadata_sheet

# Build Metadata DataFrames (DRT + Zarc fixed parameters; applied to all conditions)
df_meta_drt = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage3_drt",
    params = {
        "DRT_CV_TYPE":      DRT_CV_TYPE,
        "DRT_RBF_DER":      DRT_RBF_DER,
        "DRT_SHAPE_S":      DRT_SHAPE_S,
        "DRT_LAMBDA":       DRT_LAMBDA,
        "PEAK_HEIGHT_FRAC": PEAK_HEIGHT_FRAC,
        "PEAK_MIN_DIST_DECADES": PEAK_MIN_DIST_DECADES,
        "reference":        "pyDRTtools (Wan, Ciucci et al., 2015); relaXIS manual §7.2",
    },
)
df_meta_fit = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage3_fit",
    params = {
        "ZARC_R_DEC":      ZARC_R_DEC,
        "ZARC_TAU_DEC":    ZARC_TAU_DEC,
        "ZARC_ALPHA_INIT": ZARC_ALPHA_INIT,
        "circuit":         "R0 - Zarc_1 - ... - Zarc_N",
        "C_eff_formula":   "C_eff = Q^(1/alpha) * R^((1-alpha)/alpha)",
        "reference":       "Cole-Cole 1941; Boukamp 1986; Vendrell & West 2018",
    },
)

_export_mode = f"merged T={FOCUS_T}°C" if FOCUS_T is not None else "full overwrite"

for condition, data in all_results.items():
    results_dir = sample_dir / "Results" / condition
    results_dir.mkdir(parents=True, exist_ok=True)

    if data["drt_peaks"]:
        drt_path = results_dir / "stage3_drt.xlsx"
        df_pk = merge_sheet_by_T(drt_path, "Peaks",       pd.DataFrame(data["drt_peaks"]),   FOCUS_T)
        df_sm = merge_sheet_by_T(drt_path, "Summary",     pd.DataFrame(data["drt_summary"]), FOCUS_T)
        df_sp = merge_sheet_by_T(drt_path, "DRT_Spectra", pd.DataFrame(data["drt_spectra"]), FOCUS_T)
        with pd.ExcelWriter(drt_path, engine="openpyxl") as w:
            df_pk.to_excel(w,       sheet_name="Peaks",       index=False)
            df_sm.to_excel(w,       sheet_name="Summary",     index=False)
            df_sp.to_excel(w,       sheet_name="DRT_Spectra", index=False)
            df_meta_drt.to_excel(w, sheet_name="Metadata",    index=False)
        print(f"  [{condition}] stage3_drt.xlsx  ({len(df_sp)} spectral points)  [{_export_mode}]")

    if data["fit_peaks"]:
        fit_path = results_dir / "stage3_fit.xlsx"
        df_fp = merge_sheet_by_T(fit_path, "Peaks",   pd.DataFrame(data["fit_peaks"]),   FOCUS_T)
        df_fs = merge_sheet_by_T(fit_path, "Summary", pd.DataFrame(data["fit_summary"]), FOCUS_T)
        with pd.ExcelWriter(fit_path, engine="openpyxl") as w:
            df_fp.to_excel(w,       sheet_name="Peaks",    index=False)
            df_fs.to_excel(w,       sheet_name="Summary",  index=False)
            df_meta_fit.to_excel(w, sheet_name="Metadata", index=False)
        n_ok = int(df_fs["converged"].sum()) if "converged" in df_fs.columns else 0
        print(f"  [{condition}] stage3_fit.xlsx   ({n_ok}/{len(df_fs)} converged)  [{_export_mode}]")

print("\nExport complete.")
print("→ Next: 04_plots.ipynb")

**Next step:** run [stage4_plots.ipynb](stage4_plots.ipynb)